# Imports

In [ ]:
import random

import os

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import roc_auc_score

# Setup

In [ ]:
# Sets random seeds for reproducibility
def set_seed(seed):
    # Set seed for PyTorch
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        # Set seed for all available GPUs
        torch.cuda.manual_seed_all(seed)

    # Set seed for NumPy
    np.random.seed(seed)

    # Set seed for Python's built-in random module
    random.seed(seed)

    # Enable deterministic algorithms for reproducibility
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Use deterministic algorithms where available
    torch.use_deterministic_algorithms(True)

# Set random seeds
my_seed = 42
set_seed(my_seed)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive_root = '/content/drive'
drive.mount(drive_root)

In [ ]:
# Copy Market dataset from Google Drive to Colag
!cp "/content/drive/MyDrive/COMP-SCI 5567 - Deep Learning/Project/raw_data/market-1501.zip" "/content/Market.zip"

In [ ]:
# Unzip Market.zip
!unzip "/content/Market.zip"

# Dataset

In [ ]:
def build_id_dict(root_dir):
    id_to_images = defaultdict(list)

    for img_name in os.listdir(root_dir):
        if img_name.endswith(".jpg"):
            pid = img_name.split("_")[0]
            if pid == "-1":
                continue

            id_to_images[pid].append(os.path.join(root_dir, img_name))

    return id_to_images

In [ ]:
def split_ids(id_to_images, val_ratio=0.2):
    pids = list(id_to_images.keys())
    random.shuffle(pids)

    split = int(len(pids) * val_ratio)

    val_pids = set(pids[:split])
    train_pids = set(pids[split:])

    return train_pids, val_pids

In [ ]:
class TripletMarket1501(Dataset):
    def __init__(self, id_to_images, allowed_pids, transform=None):
        self.transform = transform

        self.id_to_images = {
            pid: imgs for pid, imgs in id_to_images.items()
            if pid in allowed_pids
        }

        self.pids = list(self.id_to_images.keys())

    def _get_anchor_positive(self):
        pid = random.choice(self.pids)
        imgs = self.id_to_images[pid]

        if len(imgs) < 2:
            return self._get_anchor_positive()

        return random.sample(imgs, 2), pid

    def _get_negative(self, exclude_pid):
        neg_pid = random.choice(self.pids)
        while neg_pid == exclude_pid:
            neg_pid = random.choice(self.pids)

        img = random.choice(self.id_to_images[neg_pid])
        return img

    def __getitem__(self, idx):
        (a_path, p_path), pid = self._get_anchor_positive()

        n_path = self._get_negative(pid)

        a = Image.open(a_path).convert("RGB")
        p = Image.open(p_path).convert("RGB")
        n = Image.open(n_path).convert("RGB")

        if self.transform:
            a = self.transform(a)
            p = self.transform(p)
            n = self.transform(n)

        return a, p, n

    def __len__(self):
        return sum(len(v) for v in self.id_to_images.values())

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((128, 64)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),

    transforms.RandomResizedCrop((128, 64), scale=(0.8, 1.0)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

val_transform = transforms.Compose([
    transforms.Resize((128, 64)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

In [ ]:
root_dir = "/content/Market-1501-v15.09.15/bounding_box_train"

id_to_images = build_id_dict(root_dir)
train_pids, val_pids = split_ids(id_to_images, val_ratio=0.2)

In [ ]:
train_dataset = TripletMarket1501(
    id_to_images=id_to_images,
    allowed_pids=train_pids,
    transform=train_transform
)

In [ ]:
val_dataset = TripletMarket1501(
    id_to_images=id_to_images,
    allowed_pids=val_pids,
    transform=val_transform
)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=True)

In [ ]:
class Market1501TestDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.root_dir = root_dir

        self.imgs = []

        for img_name in os.listdir(root_dir):
            if img_name.endswith(".jpg"):
                pid = img_name.split("_")[0]
                if pid == "-1":
                    continue

                self.imgs.append((os.path.join(root_dir, img_name), pid))

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_path, pid = self.imgs[idx]

        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, pid, img_path

In [ ]:
query_dataset = Market1501TestDataset(
    root_dir="/content/Market-1501-v15.09.15/query",
    transform=val_transform
)

query_loader = DataLoader(
    query_dataset,
    batch_size=64,
    shuffle=False
)

In [ ]:
gallery_dataset = Market1501TestDataset(
    root_dir="/content/Market-1501-v15.09.15/bounding_box_test",
    transform=val_transform
)

gallery_loader = DataLoader(
    gallery_dataset,
    batch_size=64,
    shuffle=False
)

# Model

In [ ]:
class Siamese_Network(nn.Module):
    def __init__(self):
        super(Siamese_Network, self).__init__()

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(128, 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.fc1 = nn.Linear(128, 256)
        self.fc2 = nn.Linear(256, 256)

    def forward_one(self, x):
        x = F.relu(self.pool(self.conv1(x)))   # 16 → 8
        x = F.relu(self.pool(self.conv2(x)))   # 8 → 4
        x = F.relu(self.conv3(x))              # 4 → 4

        x = self.gap(x)                        # 4×4 → 1×1
        x = x.view(x.size(0), -1)              # (B, 128)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

    def forward(self, input1, input2):
        return self.forward_one(input1), self.forward_one(input2)

# Train

In [ ]:
# Set device to GPU if available
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
# Config

# Model
model = Siamese_Network()
model.to(device) # Move model to device

# Training
num_epochs = 60
num_workers = 2
patience = 10 # Stop if no improvement after
min_delta = 0.003 # Amt increase in AUC needed for action
num_batches = -1 # Use for testing. For training with all batches use -1

# Loss function
# criterion = torch.nn.CosineEmbeddingLoss(margin=0.5)
criterion = torch.nn.TripletMarginLoss(margin=0.3)

# Optimizer
learning_rate = 0.0003
betas = (0.9, 0.999)
weight_decay = 0.0001

# Learning rate scheduler
step_size = 20
gamma = 0.1

# Saves
best_model_filepath = "/content/drive/MyDrive/COMP-SCI 5567 - Deep Learning/Project/similarity_model/best_model"
checkpoint_filepath = "/content/drive/MyDrive/COMP-SCI 5567 - Deep Learning/Project/similarity_model/training_checkpoints"

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    betas=betas,
    weight_decay=weight_decay
    )

lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=step_size,
    gamma=gamma
    )

In [ ]:
# Saves model
def save_model(model, filepath="/content/", model_name="model"):
  torch.save(model.state_dict(), f"{filepath}/{model_name}_state_dict.pth") # Saves state dict
  # torch.save(model, f"{filepath}/{model_name}.pth") # Saves entire model
  print("Model saved!")

In [ ]:
def save_checkpoint(epoch, model_state_dict, optimizer_state_dict, loss, results, filepath):
    state = {
        'epoch': epoch,
        'model_state_dict': model_state_dict,
        'optimizer_state_dict': optimizer_state_dict,
        'loss': loss,
        'results': results,
    }
    torch.save(state, filepath)
    print(f"Checkpoint saved to {filepath}")

In [ ]:
@torch.no_grad()
def compute_top1(model, query_loader, gallery_loader, device):
    model.eval()

    gallery_embs = []
    gallery_pids = []

    # Build gallery embeddings
    for imgs, pids, _ in gallery_loader:
        imgs = imgs.to(device)

        emb = model.forward_one(imgs)
        emb = F.normalize(emb, dim=1)

        gallery_embs.append(emb)
        gallery_pids.extend(pids)

    gallery_embs = torch.cat(gallery_embs, dim=0)  # keep on GPU

    correct = 0
    total = 0

    # Evaluate queries
    for imgs, pids, _ in query_loader:
        imgs = imgs.to(device)

        emb = model.forward_one(imgs)
        emb = F.normalize(emb, dim=1)

        # cosine similarity (since normalized)
        sims = emb @ gallery_embs.T

        top1_idx = sims.argmax(dim=1)

        for i in range(len(pids)):
            if gallery_pids[top1_idx[i]] == pids[i]:
                correct += 1
            total += 1

    return correct / max(total, 1)

In [ ]:
@torch.no_grad()
def validation(model, val_loader, criterion, device, epoch, num_epochs):
    model.eval()

    total_loss = 0.0
    total_samples = 0

    correct_triplets = 0

    pos_sims = []
    neg_sims = []

    pbar = tqdm(
        val_loader,
        desc=f"Epoch {epoch+1}/{num_epochs} [Val]",
        leave=False,
        dynamic_ncols=True
    )

    for anchor, positive, negative in pbar:

        anchor = anchor.to(device)
        positive = positive.to(device)
        negative = negative.to(device)

        batch_size = anchor.size(0)

        # Forward pass
        emb_a = model.forward_one(anchor)
        emb_p = model.forward_one(positive)
        emb_n = model.forward_one(negative)

        # Normalize (VERY IMPORTANT for cosine behavior)
        emb_a = F.normalize(emb_a, dim=1)
        emb_p = F.normalize(emb_p, dim=1)
        emb_n = F.normalize(emb_n, dim=1)

        # Triplet loss
        loss = criterion(emb_a, emb_p, emb_n)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        # Similarities for logging
        sim_pos = F.cosine_similarity(emb_a, emb_p, dim=1)
        sim_neg = F.cosine_similarity(emb_a, emb_n, dim=1)

        pos_sims.append(sim_pos.cpu())
        neg_sims.append(sim_neg.cpu())

        # Triplet accuracy: pos should be closer than neg
        correct_triplets += (sim_pos > sim_neg).sum().item()

        pbar.set_postfix({
            "loss": total_loss / total_samples
        })

    avg_loss = total_loss / max(total_samples, 1)
    triplet_acc = correct_triplets / max(total_samples, 1)

    # Aggregate stats
    pos_sims = torch.cat(pos_sims)
    neg_sims = torch.cat(neg_sims)

    mean_pos = pos_sims.mean().item()
    mean_neg = neg_sims.mean().item()
    gap = mean_pos - mean_neg

    print(f"Val stats → pos: {mean_pos:.4f}, neg: {mean_neg:.4f}, gap: {gap:.4f}")
    print(f"Triplet accuracy: {triplet_acc:.4f}")

    return avg_loss, triplet_acc

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, device, epoch, num_epochs, num_batches=-1):
    model.train()

    total_loss = 0.0
    total_samples = 0

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{num_epochs} [Train]",
        leave=False,
        dynamic_ncols=True
    )

    for i, (anchor, positive, negative) in enumerate(pbar):

        anchor = anchor.to(device)
        positive = positive.to(device)
        negative = negative.to(device)

        batch_size = anchor.size(0)

        # Forward pass
        emb_a = model.forward_one(anchor)
        emb_p = model.forward_one(positive)
        emb_n = model.forward_one(negative)

        # Normalize (important for cosine-based embedding stability)
        emb_a = F.normalize(emb_a, dim=1)
        emb_p = F.normalize(emb_p, dim=1)
        emb_n = F.normalize(emb_n, dim=1)

        # Triplet loss
        loss = criterion(emb_a, emb_p, emb_n)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        avg_loss = total_loss / total_samples

        pbar.set_postfix({
            "loss": avg_loss
        })

        if num_batches != -1 and i >= num_batches:
            break

    return total_loss / max(total_samples, 1)

In [ ]:
# Training loop
training_losses = []
val_losses = []
val_aucs = []
best_val_auc = 0
epochs_completed = 0
last_model_saved_epoch = 0
epochs_no_improve = 0

for epoch in range(num_epochs):
    # Train
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch, num_epochs)
    training_losses.append(train_loss)

    # Step the LR scheduler
    lr_scheduler.step()

    # Validation
    val_loss, val_auc = validation(model, val_loader, criterion, device, epoch, num_epochs)
    val_losses.append(val_loss)
    val_aucs.append(val_auc)

    epochs_completed += 1
    print(f"Epoch {epoch+1} completed | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | Val auc: {val_auc:.4f}")

    if (epoch + 1) % 5 == 0:
        top1 = compute_top1(model, query_loader, gallery_loader, device)
        print(f"Top-1 Accuracy: {top1:.4f}")

    save_checkpoint(
        epoch=epochs_completed,
        model_state_dict=model.state_dict(),
        optimizer_state_dict=optimizer.state_dict(),
        loss=val_loss,
        results=val_auc,
        filepath=f"{checkpoint_filepath}/save_checkpoint_{epochs_completed}.pth"
        )

    # Model save if better than before
    if val_auc > best_val_auc + min_delta:
      best_val_auc = val_auc
      epochs_no_improve = 0
      save_model(model=model, filepath=best_model_filepath, model_name="similarity_model")
      last_model_saved_epoch = epoch + 1
    else: # Early stopping if no improvements
      epochs_no_improve += 1
      if epochs_no_improve >= patience:
        print(f"Training stopped early at epoch {epochs_completed}")
        break
      else:
        print(f"No improvement, early stopping may happen after {patience - epochs_no_improve} more epochs if no improvements")

In [ ]:
# Plots training and validation losses
sns.lineplot(x=range(1, len(training_losses) + 1), y=training_losses, label="Training Loss")
sns.lineplot(x=range(1, len(val_losses) + 1), y=val_losses, label="Validation Loss")

plt.title("Training vs Validation Loss Over Epochs")
plt.xticks(range(1, len(val_losses) + 1))
plt.axvline(x=last_model_saved_epoch, color='r', linestyle='--', label='Best model saved')
plt.legend()
plt.show()

In [ ]:
# Plots training and validation accs
sns.lineplot(x=range(1, len(val_aucs) + 1), y=val_aucs, label="Validation AUC")

plt.title("Validation AUC Over Epochs")
plt.xticks(range(1, len(val_aucs) + 1))
plt.axvline(x=last_model_saved_epoch, color='r', linestyle='--', label='Best model saved')
plt.legend()
plt.show()

In [ ]:
from google.colab import runtime
runtime.unassign()